# Full-Trajectory Training with Negative Log-Likelihood (NLL) Loss

This notebook demonstrates how to train a model on the Lorenz system using the negative log-likelihood (NLL) loss over the entire trajectory, without batching or windowing.

## 1. Import Required Libraries

Import all necessary Julia packages for simulation, optimization, and plotting.

In [1]:
# Activate project and import packages
import Pkg
Pkg.activate("./LorenzParameterEstimation")
Pkg.instantiate()

using LorenzParameterEstimation
using DifferentialEquations
using Random
using LinearAlgebra
using Statistics
using Plots

  Activating project at `~/master_thesis/LorenzParameterEstimation`
Precompiling project...
   1945.3 ms  ✓ LorenzParameterEstimation
   6057.5 ms  ✓ LorenzParameterEstimation → LorenzVisualizationExt
  2 dependencies successfully precompiled in 11 seconds. 612 already precompiled.


## 2. Load and Prepare Data

Simulate the Lorenz system to generate a full trajectory. No batching or windowing is used; the entire trajectory will be used for training.

In [14]:
# Set random seed for reproducibility
Random.seed!(42)

# Define true parameters and initial condition
true_params = classic_params()
u0 = [1.0, 1.0, 1.0]
T = 100.0
M = 20_000

dt = T / M

# Simulate the true trajectory
sol_true = integrate(true_params, u0, (0.0, T), dt)

# The full trajectory is now in sol_true.u (size: M x 3)


L63Solution{Float64, Vector{Float64}, Matrix{Float64}, L63System{Float64, Vector{Float64}}}([0.0, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03, 0.035, 0.04, 0.045  …  99.955, 99.96000000000001, 99.965, 99.97, 99.97500000000001, 99.98, 99.985, 99.99000000000001, 99.995, 100.0], [1.0 1.0 1.0; 1.0031933311430772 1.129839801362649 0.9920510551862709; … ; -0.9938824148762829 -1.892609128103656 16.965693824136157; -1.0388569689855993 -1.9396559790447456 16.750658939956015], L63System{Float64, Vector{Float64}}(L63Parameters{Float64}(10.0, 28.0, 2.6666666666666665, 0.0, 0.0, 0.0, 1.0), [1.0, 1.0, 1.0], (0.0, 100.0), 0.005), [-1.0388569689855993, -1.9396559790447456, 16.750658939956015], true)

## 3. Define and Train the Model

Define the negative log-likelihood (NLL) loss and optimize parameters using the full trajectory. No batching or windowing is performed.

In [15]:
# Define the negative log-likelihood (NLL) loss for Gaussian noise
function gaussian_nll(x_true, x_pred, σ2)
    n = length(x_true)
    return sum(0.5 * log(2π*σ2) .+ 0.5 * (x_true .- x_pred).^2 / σ2) / n
end

# Initial guess for parameters (e.g., shifted x_s)
init_params = with_coordinate_shifts(true_params, 10.0, 0.0, 0.0)



L63Parameters{Float64}(10.0, 28.0, 2.6666666666666665, 10.0, 0.0, 0.0, 1.0)

In [16]:
# Optimization settings
σ2 = 1.0  # assumed noise variance
learning_rate = 1e-5  # much smaller learning rate
n_epochs = 200
x_s_min, x_s_max = -50.0, 50.0  # clamp range

# Store parameter history
x_s_history = Float64[]
loss_history = Float64[]

params = deepcopy(init_params)

for epoch in 1:n_epochs
    # Simulate model trajectory with current parameters
    sol_pred = integrate(params, u0, (0.0, T), dt)
    x_pred = sol_pred.u[:, 1]  # predict x coordinate
    x_true = sol_true.u[:, 1]

    # Check for NaN/Inf in ODE output
    if any(isnan, x_pred) || any(isinf, x_pred)
        println("ODE integration failed: NaN or Inf in x_pred at epoch $epoch, x_s=$(params.x_s)")
        break
    end

    # Compute NLL loss
    loss = gaussian_nll(x_true, x_pred, σ2)

    # Compute gradient w.r.t. x_s (coordinate shift)
    x_s_eps = params.x_s + 1e-5
    sol_pred_eps = integrate(with_coordinate_shifts(params, x_s_eps, 0.0, 0.0), u0, (0.0, T), dt)
    x_pred_eps = sol_pred_eps.u[:, 1]
    if any(isnan, x_pred_eps) || any(isinf, x_pred_eps)
        println("ODE integration failed: NaN or Inf in x_pred_eps at epoch $epoch, x_s=$x_s_eps")
        break
    end
    grad = (gaussian_nll(x_true, x_pred_eps, σ2) - loss) / 1e-5

    # Check for NaN or Inf in loss/grad
    if isnan(loss) || isnan(grad) || isinf(loss) || isinf(grad)
        println("NaN or Inf detected! loss=$loss, grad=$grad, x_s=$(params.x_s)")
        break
    end

    # Gradient descent update (only x_s for demonstration)
    new_x_s = clamp(params.x_s - learning_rate * grad, x_s_min, x_s_max)
    params = with_coordinate_shifts(params, new_x_s, 0.0, 0.0)

    push!(x_s_history, params.x_s)
    push!(loss_history, loss)

    println("Epoch $epoch: loss = $loss, x_s = $(params.x_s)")
end

Epoch 1: loss = 137.62285735736677, x_s = -9.823795498289144
Epoch 2: loss = 108.10118508892009, x_s = -5.099800345969129
Epoch 3: loss = 75.3386915239118, x_s = 0.4957602723662262
Epoch 4: loss = 55.381729573384504, x_s = -1.056754642560669
Epoch 5: loss = 51.24990461402753, x_s = -6.905589194469158
Epoch 6: loss = 74.30987739425267, x_s = -14.548655535078211
Epoch 7: loss = 168.54002387103964, x_s = -4.855961305222763
Epoch 8: loss = 68.44174505724757, x_s = -4.6851122955829965
Epoch 9: loss = 61.57682554598605, x_s = -4.79552035944392
Epoch 10: loss = 66.23551865325834, x_s = -5.745329071953229
Epoch 11: loss = 62.04390125733211, x_s = -6.019918929156454
Epoch 12: loss = 72.57209518017557, x_s = 2.941902789302951
Epoch 13: loss = 85.94113926763885, x_s = -1.5347488616349736
Epoch 14: loss = 46.53620181661307, x_s = -5.347909171128165
Epoch 15: loss = 74.76527296768982, x_s = -0.09564879141507188
Epoch 16: loss = 61.243205633643285, x_s = 4.200633117157701
Epoch 17: loss = 92.1298235

## 4. Evaluate Model Performance

Visualize the training loss and the evolution of the estimated parameter.

In [17]:
# Plot training loss and parameter evolution
plot(loss_history, label="NLL Loss", xlabel="Epoch", ylabel="Loss", title="Training Loss Over Epochs")

plot(x_s_history, label="Estimated x_s", xlabel="Epoch", ylabel="x_s", title="x_s Evolution")

println("Final estimated x_s: ", params.x_s)
println("True x_s: 0.0 (since we started from classic_params)")

Final estimated x_s: 30.6045705708583
True x_s: 0.0 (since we started from classic_params)
